<a href="https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

It is the multi-class classification lane because the web pages are going to be classified as `stable`, `up` and `down`. The target (`trend_direction`) is a set of predefined, unordered, discrete labels.

The clustering problem is about similarity based grouping with no predefined labels. But there are defined categories.

Ranking does the relative comparison between items. `stable`, `up` and `down` are nominal categories with no inherent order or magnitude between them as opposed to ordinals.


Scoring/regression produces the continous number but here the target variables are discrete.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The `trend_direction` is a proxy label come from the defined rules to raw metrices like impressions and clicks over a specified time window.

There are 5 nominal labels `stable`, `up`, `down`, `new` and `flat`. In this session, we will going for 3 class classification problem discarding the 2 labels `flat` and `new`. They required another classification model to predict which items started to gain the visbility or which items will remain invisible. That's a binary classification problem.

Another caveat is both labels look like structural, zero-history artifacts rather than genuine behavoral trends. That's a proxy label weakness worth stating plainly - these labels needs to be verified.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: Precision@K**

Plain accuracy is misleading here because the target classes are imbalanced
(`stable` likely dominates the dataset). A model could predict "stable" for
almost everything and still score high on accuracy, without ever correctly
flagging a real decline — a majority-class baseline masquerading as a good
model.

The actual use case is a human review queue: a reviewer only looks at the
top K pages the model flags, not all ~30,000. Precision@K measures what
fraction of those top-K flagged pages are genuinely declining, which is the
only thing that determines whether the shortlist is trustworthy.

Precision@K alone doesn't fully account for the false-negative cost
asymmetry (missing a real decline is worse than a false alarm) — it only
tells you the queue's quality, not how many real declines were missed
entirely. Recall (or recall@K) is a secondary metric worth tracking
alongside precision@K to catch that gap, but precision@K is the primary,
defensible number for this review-queue use case.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = One content page (`content_id`), and each of those pages belong to one of 32 unique clients.

In [1]:
!git clone https://github.com/Hadeed07/FlyRank-ML.git
import pandas as pd


df = pd.read_csv('/content/FlyRank-ML/data/raw/content_refresh_anonymized.csv')
print(df.shape)
print(df['content_id'].nunique())   # should match row count if truly 1 row per page
print(df['client_id'].nunique())    # should be 32
df.head()

Cloning into 'FlyRank-ML'...
remote: Enumerating objects: 219, done.
remote: Counting objects: 100% (219/219), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 219 (delta 115), reused 104 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (219/219), 2.03 MiB | 7.88 MiB/s, done.
Resolving deltas: 100% (115/115), done.
(30000, 44)
30000
32


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
df[['content_id', 'client_id', 'trend_direction']].head(10)

,content_id,client_id,trend_direction
0,content_304f48230142,client_f369cb89fc,down
1,content_a1fb4e703a9e,client_4e07408562,down
2,content_9aa793d4d895,client_7f2253d7e2,down
3,content_331d6c4de07b,client_19581e27de,stable
4,content_d99b7a2d90ca,client_3fdba35f04,down
5,content_d4084a4bc775,client_f369cb89fc,down
6,content_9a34b442b552,client_8722616204,down
7,content_a63219c6e95a,client_19581e27de,stable
8,content_5e6c160719bc,client_6208ef0f77,down
9,content_c27558df2b0c,client_19581e27de,down


Each row above confirms: one row = one content page (`content_id`), belonging to one `client_id`, with `trend_direction` as its target label. The label is categorical, not numeric — `down`, `stable`, `up` (plus `new`/`flat`, which will be excluded per Section 2). No two rows share a `content_id`, so there's no page-level duplication to worry about; the only grouping concern is at the `client_id` level (32 clients), which is a train/test split issue, not a row-definition issue.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The problem isn't the volume of data (30,000 rows isn't what break if-statements). A computer can iterate over those rows but the bottleneck is adaptive thresholding based on context. A fixed rule can't adapt its threshold.


A low prev_30d_impressions means something different depending on content_age_days (new page vs. old page) and competition (high-competition niche vs. low). To hand-write that with if-statements, you'd need a separate branch for every combination of your ~9 features — and those combinations interact in ways you can't fully anticipate or enumerate by hand.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.